# Session 3 - VLM smoke matrix


**GPU T4 x2, ~1 h.** Loads each candidate VLM once and checks the four properties that
must hold before any GPU time is committed: probabilities in range, citations normalised
over the vocabulary, determinism, and measured throughput.

The samples-per-GPU-hour figure printed here is the basis for choosing `VLM_SAMPLES`
in Session 5.

In [ ]:
SESSION = "S3 VLM smoke"

# ============================== CONFIG ==============================
MODELS = [
    "qwen25vl:Qwen/Qwen2.5-VL-3B-Instruct",
    "hfvlm:Qwen/Qwen2-VL-2B-Instruct",
    "hfvlm:HuggingFaceTB/SmolVLM-Instruct",
    # "qwen25vl:Qwen/Qwen2.5-VL-7B-Instruct",   # needs QUANT="4bit"
]
QUANT      = "fp16"      # auto | fp16 | bf16 | 4bit
MAX_PIXELS = 384 * 384   # ~190 visual tokens at a 384 px crop
N_SMOKE    = 4           # real crops per model

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
KU.pip_install("transformers accelerate qwen-vl-utils")
if QUANT == "4bit":
    KU.pip_install("bitsandbytes")
KU.gpu_report()

In [ ]:
INDEX = KU.find_parsed_index()
print("INDEX =", INDEX or "(none: the smoke test will use random images)")

In [ ]:
# ==================== SMOKE MATRIX ====================
idx = f' --index "{INDEX}"' if INDEX else ""
results = []
for spec in MODELS:
    print(C.banner(spec))
    KU.sh(f'{sys.executable} -m ccaudit.m4_detectors --smoke-vlm "{spec}" '
          f'--quant {QUANT} --max-pixels {MAX_PIXELS} --n {N_SMOKE}{idx} '
          f'--out "{OUT}/smoke/{C.safe_name(spec)}.json"',
          check=False, log=f"{OUT}/logs/smoke.log")

In [ ]:
# ==================== BUDGET TABLE ====================
import glob
rows = []
for p in sorted(glob.glob(f"{OUT}/smoke/*.json")):
    for r in C.load_json(p).get("results", []):
        rows.append(r)
C.save_json(f"{OUT}/smoke/smoke_matrix.json", {"results": rows}, indent=1)

print(f"{'model':46s}{'load s':>8}{'s/call':>9}{'samp/GPU-h':>12}  status")
for r in rows:
    st = "OK" if r.get("ok") else "FAILED: " + "; ".join(r.get("errors", []))[:60]
    print(f"{r['spec'][:45]:46s}{r.get('load_sec',0):>8.0f}"
          f"{r.get('mean_sec_per_call',float('nan')):>9.3f}"
          f"{r.get('samples_per_gpu_hour',float('nan')):>12.0f}  {st}")

print("\nSizing Session 5 (2 GPUs, 43 calls/sample with PROMPT_VARIANTS=5):")
for r in rows:
    sph = r.get("samples_per_gpu_hour") or float("nan")
    if sph == sph and sph > 0:
        for hrs in (2, 4, 7):
            print(f"  {r['spec'][:40]:42s} {hrs} GPU-h on 2 cards "
                  f"-> ~{2*hrs*sph*35/43:.0f} samples")
        break

In [ ]:
NEXT_STEP = """1. Any model marked FAILED is excluded from the model set; record the reason.
2. Choose VLM_SAMPLES for Session 5 from the sizing table (600 is the default
   main size; 300 for the 7B model).
3. Output tab -> New Dataset -> `cca-s3-smoke` (small; keeps the throughput
   numbers with the rest of the provenance).
4. Continue with Session 4 (CNN) or directly with Session 5 (VLM main audit)."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)